In [9]:
import pandas as pd
from sklearn.cluster import KMeans
import numpy as np
from collections import defaultdict
from scipy.spatial.distance import cdist
from copy import deepcopy
import itertools
import heapq
import random
from itertools import chain
import copy
import math
from itertools import combinations
import time
import matplotlib.pyplot as plt
import seaborn as sns
from gurobipy import Model, GRB, quicksum

In [11]:
file_name = 'CMT1.txt'
if file_name == 'CMT1.txt':
    big_Q = 160
    v_value = 5 
    n = 50
    r = 3
if file_name == 'CMT2.txt':
    big_Q = 140
    v_value = 10
    n = 75
    r = 4
if file_name == 'CMT3.txt':
    big_Q = 200
    v_value = 8
    n = 100
    r = 5
if file_name == 'CMT4.txt':
    big_Q = 200
    v_value = 12
    n = 75
    r = 4
if file_name == 'CMT5.txt':
    big_Q = 200
    v_value = 16
    n = 100
    r = 5
locations = {}
customer_loads = {}
customer_coords_dict = {}
v = v_value
with open(file_name, "r") as file:
    lines = file.readlines()
index = lines.index("NODE_COORD_SECTION\n") + 1  # Find start of section
for i in range(index, index + n + 1):
    parts = lines[i].strip().split()
    node_id = int(parts[0]) - 1  # Adjust index to start from 0
    x_coord = float(parts[1])
    y_coord = float(parts[2])
    locations[node_id] = (x_coord, y_coord)
    # Create a dictionary of customer coordinates
    if node_id != 0:
        customer_coords_dict[node_id] = (x_coord, y_coord) #The keys are customer IDs (c in C).The values are tuples (x, y), representing the customer's coordinates.


# Process demand section
index = lines.index("DEMAND_SECTION\n") + 1  # Find start of section
for i in range(index, index + n + 1):
    parts = lines[i].strip().split()
    node_id = int(parts[0]) - 1  # Adjust index to start from 0
    demand = float(parts[1])
    customer_loads[node_id] = demand

# Print results
del customer_loads[0]

C = list(range(1,n+1))   # set of customers
N = list(range(0,n+1)) # Nodes in the network
TW = [] # earliest and latest time to serve a customer for each customer
V=list(range(1,v+1))
for i in N:
    TW.append((2, 600000000*8))

# Create a DataFrame for better visualization
traveling_time = {}
distance  = {}

# Compute Euclidean distance between each pair of nodes
for i, (x1, y1) in locations.items():
    for j, (x2, y2) in locations.items():
        distance_val = math.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)
        traveling_time[(i, j)] = distance_val
        distance[(i, j)] = distance_val
traveling_time[0, n + 1] = 0
traveling_time[0, 0] = 0
for i in C:
    traveling_time[i,n+1] = traveling_time[i,0] 

In [23]:
outcome = []
for iter_val in [int(0.4*big_Q),int(0.6*big_Q),int(0.8*big_Q)]:
    k = v
    K = list(range(k))  # Set of vehicles
    Ck = {i: iter_val for i in K}  # Capacity of vehicle k
    TH = 4*600000  # Planning horizon
    M = 100000  # Large constant

    # Define data (Placeholder: replace with actual data)
    C = list(range(1,n+1))  # Set of customers
    R = list(range(r))  # Set of trips
    N_prime = [0] + C  # Nodes including depot (0)
    N_double_prime = C + [n+1]  # Nodes including depot and end node (n+1)
    NTot = [0] + C + [n+1]

    # Create model
    model = Model("Vanilla Multi-Trip VRP")

    # Decision variables
    y = model.addVars(C, K, R, vtype=GRB.BINARY, name="y")
    x = model.addVars(N_prime, N_double_prime, K, R, vtype=GRB.BINARY, name="x")

    arrival = model.addVars(C, K, R, vtype=GRB.CONTINUOUS, name="arrival")
    start = model.addVars(K, R, vtype=GRB.CONTINUOUS, name="start")
    end = model.addVars(K, R, vtype=GRB.CONTINUOUS, name="end")

    # Objective function: Maximize profit
    model.setObjective(quicksum(distance[i,j]*x[i,j,k,r] for k in K for r in R for i in N_prime for j in N_double_prime if i!=j), GRB.MINIMIZE)

    # Constraints
    # (2) Each customer served once (fleet or third-party)
    model.addConstrs((quicksum(y[i, v, r] for v in K for r in R) == 1 for i in C), name="serve_once")

    # (3) routing
    model.addConstrs((quicksum(x[j, i, v, r] for j in N_prime) == quicksum(x[i, j, v, r] for j in N_double_prime) for i in C for v in K for r in R), name="flow_out")
    model.addConstrs((quicksum(x[i, j, v, r] for i in N_prime if i!=j) == y[j, v, r] for j in C for v in K for r in R), name="flow_out")
    model.addConstrs((quicksum(x[i, j, v, r] for j in N_double_prime if j!=i) == y[i, v, r] for i in C for v in K for r in R), name="flow_in")
    model.addConstrs((quicksum(x[j,n+1, v, r] for j in C) == quicksum(x[0,j, v, r] for j in C) for v in K for r in R), name="flow_in")
    model.addConstrs((quicksum(x[0,j, v, r] for j in N_double_prime) <= 1 for v in K for r in R), name="each trip is used at most once")


    # (5) Vehicle capacity constraints
    model.addConstrs((quicksum(customer_loads[i] * y[i, v, r] for i in C) <= Ck[v] * quicksum(x[0, j, v, r] for j in C) for v in K for r in R), name="capacity")

    # (6) Arrival time for first trip node
    model.addConstrs((arrival[j, v, r] >= start[v, r] + traveling_time[0, j] * x[0, j, v, r] - 1000000*(1- x[0, j, v, r]) for j in C for v in K for r in R if r != 0), name="arrival_first")
    model.addConstrs((arrival[j, v, 0] >= start[v, 0] + traveling_time[0, j] * x[0, j, v, 0] - 1000000*(1- x[0, j, v, 0]) for j in C for v in K), name="arrival_first")

    # (7) and (8) Arrival time between consecutive nodes
    model.addConstrs((arrival[j, v, r] >= arrival[i, v, r] + traveling_time[i, j] * x[i, j, v, r] - M * (1 - x[i, j, v, r]) for i in C for j in C if i != j for v in K for r in R), name="arrival_lower")
    model.addConstrs((arrival[j, v, r] <= arrival[i, v, r] + traveling_time[i, j] * x[i, j, v, r] + M * (1 - x[i, j, v, r]) for i in C for j in C if i != j for v in K for r in R), name="arrival_upper")
    # (9) Deadline constraints

    # (10) and (11) End of trip constraints
    model.addConstrs((end[v, r - 1] <= start[v, r] for v in K for r in R if r != 0), name="recharge_between")
    model.addConstrs((end[v, r] >= arrival[i, v, r] + traveling_time[i, n+1] * x[i, n+1, v, r] - M * (1 - x[i, n+1, v, r]) for i in C for v in K for r in R), name="end_lower")
    model.addConstrs((end[v, r] <= arrival[i, v, r] + traveling_time[i, n+1] * x[i, n+1, v, r] + M * (1 - x[i, n+1, v, r]) for i in C for v in K for r in R), name="end_upper")

    model.addConstrs((end[v, r]  >= start[v, r] for v in K for r in R), name="recharge_between")
    model.addConstrs((end[v, r2] <= start[v, r] for v in K for r in R if r != 0 for r2 in range(r)), name="recharge_between")
    model.addConstrs((end[v, r]  <= TH for v in K for r in R), name="time_horizon")
    # Optimize the model
    model.setParam('TimeLimit', 60*60*18)

    model.optimize()

    if model.status == GRB.TIME_LIMIT or model.status == GRB.OPTIMAL:
        if model.SolCount == 0:
            save_output =  [file_name, n, v, iter_val, 999999999,9999999,9999999,9999999, 'Gurobi']
        else:
            lower_bound = model.objBound
            upper_bound = model.ObjVal
            gap = (upper_bound - lower_bound)/lower_bound*100
            save_output =  [file_name, n, iter_val, v, lower_bound,upper_bound,gap,model.Runtime, 'Gurobi']

    outcome.append(save_output)
    df_outcome = pd.DataFrame(outcome, columns = ['Instance ID', 'N' , 'Q', 'V', 'LB' , 'UB', 'Gap','Time', 'Solver'])
    file_name_n = file_name + str(iter_val) + '_Gurobi'
    df_outcome.to_csv(f'outcome_{file_name_n}.csv')

Set parameter TimeLimit to value 64800
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (win64 - Windows 10.0 (19045.2))

CPU model: Intel(R) Xeon(R) E-2124 CPU @ 3.30GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 4 logical processors, using up to 4 threads

Non-default parameters:
TimeLimit  64800

Optimize a model with 816412 rows, 416120 columns and 4044384 nonzeros
Model fingerprint: 0x0027e962
Variable types: 4080 continuous, 412040 integer (412040 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+06]
  Objective range  [1e+00, 9e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+06]
Presolve removed 4088 rows and 4056 columns (presolve time = 5s)...
Presolve removed 4128 rows and 4056 columns (presolve time = 10s)...
Presolve removed 4088 rows and 4016 columns
Presolve time: 14.04s
Presolved: 812324 rows, 412104 columns, 3235528 nonzeros
Variable types: 4064 continuous, 408040 integer (408040 binary)

Deterministic concurrent LP

In [7]:
2*18/24

1.5